# Notebook 10 — Bioenergy Calculation

**Purpose:** Convert recoverable residue into energy potential (GJ), compressed biogas volume (CBG tonnes), and annual revenue (₹ crore), then aggregate to district level and compute a normalised Bioenergy Potential Score (BPS 0–100).

**Parameters:**
- LHV: Rice straw 14.3 GJ/t, Wheat straw 17.5 GJ/t
- CBG yield: 22 kg per tonne recoverable residue
- CBG price: ₹46/kg

**Input:** `Data/Processed/residue_calc.csv`  
**Output:** `Data/Processed/bioenergy_scores.csv`

In [2]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load residue data 
residue = pd.read_csv('../Data/Processed/residue_calc.csv')
print(f"Shape: {residue.shape}")
print(residue.head())

Shape: (611, 10)
    state  district         year  crop  production_t  RPR  \
0  Punjab  Amritsar  2010 - 2011  Rice      503000.0  1.5   
1  Punjab  Amritsar  2011 - 2012  Rice      482000.0  1.5   
2  Punjab  Amritsar  2012 - 2013  Rice      544000.0  1.5   
3  Punjab  Amritsar  2013 - 2014  Rice      518000.0  1.5   
4  Punjab  Amritsar  2014 - 2015  Rice      505000.0  1.5   

   residue_generated_t  burn_fraction  burned_t  recoverable_t  
0             754500.0            0.8  603600.0       105630.0  
1             723000.0            0.8  578400.0       101220.0  
2             816000.0            0.8  652800.0       114240.0  
3             777000.0            0.8  621600.0       108780.0  
4             757500.0            0.8  606000.0       106050.0  


In [3]:
# Lower Heating Value (LHV) → Gross Energy Potential (GEP) 
# LHV = energy available per tonne of dry biomass.
# Source: Channiwala & Parikh (2002) / FAO biomass energy values.
LHV = {'Rice': 14.3, 'Wheat': 17.5}   # GJ per tonne

residue['LHV']    = residue['crop'].map(LHV)
residue['GEP_GJ'] = residue['recoverable_t'] * residue['LHV']

print(residue[['crop', 'recoverable_t', 'LHV', 'GEP_GJ']].head())


   crop  recoverable_t   LHV     GEP_GJ
0  Rice       105630.0  14.3  1510509.0
1  Rice       101220.0  14.3  1447446.0
2  Rice       114240.0  14.3  1633632.0
3  Rice       108780.0  14.3  1555554.0
4  Rice       106050.0  14.3  1516515.0


In [4]:
# Compressed Biogas (CBG) and revenue 
# CBG yield: 22 kg per tonne recoverable residue (anaerobic digestion estimate).
# CBG price: ₹46/kg (SATAT scheme reference price, GOI).
residue['CBG_kg']     = residue['recoverable_t'] * 22
residue['revenue_Rs'] = residue['CBG_kg'] * 46

print(residue[['crop', 'CBG_kg', 'revenue_Rs']].head())


   crop     CBG_kg   revenue_Rs
0  Rice  2323860.0  106897560.0
1  Rice  2226840.0  102434640.0
2  Rice  2513280.0  115610880.0
3  Rice  2393160.0  110085360.0
4  Rice  2333100.0  107322600.0


In [5]:
# Aggregate to district × year level 
bio = residue.groupby(['state', 'district', 'year']).agg(
    NBP_GJ        = ('GEP_GJ',     'sum'),
    CBG_tonnes    = ('CBG_kg',     lambda x: x.sum() / 1000),
    revenue_crore = ('revenue_Rs', lambda x: x.sum() / 1e7),
).reset_index()

print(f"Annual district shape: {bio.shape}")
print(bio.head())


Annual district shape: (306, 6)
     state       district     year      NBP_GJ  CBG_tonnes  revenue_crore
0  Haryana         Ambala  2022-23  4825397.50    6642.790      30.556834
1  Haryana        Bhiwani  2022-23  2883245.75    3732.575      17.169845
2  Haryana  Charkhi Dadri  2022-23  1029624.75    1322.475       6.083385
3  Haryana      Faridabad  2022-23  1021000.75    1347.115       6.196729
4  Haryana      Fatehabad  2022-23  9385876.50   12665.730      58.262358


In [6]:
# Average across years to get one row per district
bio_avg = bio.groupby(['state', 'district']).agg(
    avg_NBP_GJ        = ('NBP_GJ',        'mean'),
    avg_CBG_tonnes    = ('CBG_tonnes',    'mean'),
    avg_revenue_crore = ('revenue_crore', 'mean'),
).reset_index()

# Bioenergy Potential Score (BPS) 0–100 
scaler = MinMaxScaler(feature_range=(0, 100))
bio_avg['BPS'] = scaler.fit_transform(bio_avg[['avg_NBP_GJ']]).round(1)

# Sort by revenue descending
bio_avg = bio_avg.sort_values('avg_revenue_crore', ascending=False).reset_index(drop=True)

print(f"\nTop 10 districts by bioenergy revenue:")
print(
    bio_avg[['district', 'avg_NBP_GJ', 'avg_revenue_crore', 'BPS']]
    .head(10)
    .to_string(index=False)
)



Top 10 districts by bioenergy revenue:
 district   avg_NBP_GJ  avg_revenue_crore   BPS
  Sangrur 1.386492e+07          85.274386 100.0
 Ludhiana 1.188787e+07          73.353567  85.0
   Karnal 1.053624e+07          65.819215  74.8
  Patiala 1.053433e+07          64.603110  74.7
 Bathinda 1.049216e+07          63.205982  74.4
  Kaithal 9.811013e+06          62.046985  69.3
    Sirsa 1.018777e+07          61.600693  72.1
     Jind 9.647696e+06          59.688013  68.0
Firozepur 9.635836e+06          58.757693  67.9
Fatehabad 9.385876e+06          58.262358  66.0


In [7]:
# Summary 
print(bio_avg[['avg_NBP_GJ', 'avg_CBG_tonnes', 'avg_revenue_crore', 'BPS']].describe().round(1))

       avg_NBP_GJ  avg_CBG_tonnes  avg_revenue_crore    BPS
count        45.0            45.0               45.0   45.0
mean    5777100.7          7693.8               35.4   38.7
std     3517857.9          4713.2               21.7   26.7
min      677253.5           901.7                4.1    0.0
25%     2883245.8          3732.6               17.2   16.7
50%     5291213.4          7038.6               32.4   35.0
75%     8597858.5         11323.1               52.1   60.1
max    13864917.6         18537.9               85.3  100.0


In [9]:
# Save 
bio_avg.to_csv('../Data/Processed/bioenergy_scores.csv', index=False)
print("Saved: Data/Processed/bioenergy_scores.csv")
print(f"   Rows    : {len(bio_avg)}")
print(f"   Columns : {bio_avg.columns.tolist()}")


✅ Saved: Data/Processed/bioenergy_scores.csv
   Rows    : 45
   Columns : ['state', 'district', 'avg_NBP_GJ', 'avg_CBG_tonnes', 'avg_revenue_crore', 'BPS']
